# Part 1: Cleaning Up Our House Data 🧹
Hello! In this first notebook, we will look at our house data. Sometimes data has missing pieces or weird words. We need to clean it up before we can use it!


## Looking at the Data
Let's open our file 'Hyderabad_House_Data.csv' and peek inside.


In [2]:
# First, let's bring in 'pandas'. It helps us read tables like an Excel file!
import pandas as pd
import numpy as np
import re


In [3]:
# Read the CSV file into a table called 'df' (Data Frame)
df = pd.read_csv('Hyderabad_House_Data.csv')

# Look at the first 5 rows
df.head()


,Unnamed: 0,Bedrooms,Bathrooms,Furnishing,Tennants,Area,Price,Locality
0,0,3 BHK Builder Floor,2,Furnished,Bachelors/Family,1800 sqft,"34,000","Bhagyalaxmi Nagar, Kavadiguda"
1,1,3 BHK Apartment,2,Semi-Furnished,Family,2500 sqft,"45,000","Gachibowli, Outer Ring Road"
2,2,1 BHK Builder Floor,Immediately,Furnished,Bachelors/Family,read more,"18,000",Gachibowli
3,3,3 BHK Apartment,Immediately,Furnished,Bachelors/Family,2160 sqft,"40,000","Moosapet, NH"
4,4,3 BHK Apartment,2,Semi-Furnished,Family,1580 sqft,"23,000",Raghavendra Colony kondapur


## Cleaning Step 1: Throw away empty rows
If a house doesn't have a price or an area, we can't learn from it! Let's delete those rows.


In [4]:
# Drop rows where Area, Price, or Locality are missing
df = df.dropna(subset=['Area', 'Price', 'Locality'])
print("Done dropping empty rows!")


Done dropping empty rows!


## Cleaning Step 2: Fix the Price
Some prices have symbols or commas. We just want plain numbers.


In [5]:
# Remove anything that is not a number from the Price column
df['Price'] = df['Price'].astype(str).str.replace(r'[^\d.]', '', regex=True)

# Turn it into a real number
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# Drop any that failed to turn into a number
df = df.dropna(subset=['Price'])
print("Prices are clean now!")


Prices are clean now!


## Cleaning Step 3: Make all sizes the same ('Gaj')
Some sizes are in 'sqft' (square feet) and some are in acres. Let's make them all 'gaj' (square yards). 1 gaj = 9 square feet!


In [6]:
def convert_to_gaj(text):
    # Make text lowercase and remove commas
    text = str(text).lower().replace(',', '')
    
    # Find the number inside the text
    number_match = re.search(r'(\d+[\d.]*)', text)
    if not number_match:
        return np.nan
    
    # Turn the found text into a decimal number
    try:
        size = float(number_match.group(1))
    except ValueError:
        return np.nan
        
    # Check the words to see what measurement it was
    if 'yard' in text or 'gaj' in text or 'sqyrd' in text:
        return size # Already perfect!
    elif 'acre' in text:
        return size * 4840.0 # 1 acre is 4840 gaj
    else:
        # Usually it is square feet if they don't say. Divide by 9!
        return size / 9.0

# Apply our magical function to the 'Area' column
df['Area_in_Gaj'] = df['Area'].apply(convert_to_gaj)

# Drop rows where we couldn't figure out the size
df = df.dropna(subset=['Area_in_Gaj'])
print("All areas are now in Gaj!")


All areas are now in Gaj!


## Cleaning Step 4: Simple Locality Names
Let's make all neighborhood (locality) names simple lowercase letters.


In [7]:
# Make it lowercase and strip spaces
df['Locality'] = df['Locality'].astype(str).str.lower().str.strip()

# Keep only letters and numbers (remove weird symbols)
df['Locality'] = df['Locality'].apply(lambda x: re.sub(r'[^a-z0-9\s]', '', x))


## Save the Clean Data!
Now that everything is clean, let's save it to a new file so our next notebook can use it!


In [8]:
# Save to a new CSV file
df.to_csv('Cleaned_House_Data.csv', index=False)
print("Saved successfully to Cleaned_House_Data.csv! We have", len(df), "houses ready.")


Saved successfully to Cleaned_House_Data.csv! We have 1037 houses ready.
